# Meteosat cloud animation around Maroantsetra flood dates

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johrosa/srwi/blob/main/meteosat_cloud_animation_maroantsetra_colab.ipynb)

This notebook builds short cloud-motion animations around known flood/cyclone dates near Maroantsetra, Madagascar.

For raw geostationary cloud imagery, Madagascar is covered by EUMETSAT Meteosat-9 IODC, with 15-minute observations. In Earth Engine, the scripted alternative used here is the Oya geostationary precipitation product, derived from VIS/IR geostationary observations including Meteosat-9/10.

Default events included:
- Cyclone Herold / Maroantsetra flooding: 2020-03-13 to 2020-03-18.
- Cyclone Gamane / north-east Madagascar impacts: 2024-03-26 to 2024-03-29.

Add or edit events in the parameters cell if you have field-confirmed flood dates.

## 1. Install and imports

In [ ]:
!pip -q install earthengine-api geemap requests pillow

In [ ]:
import os
import math
import requests
import ee
import geemap
import pandas as pd
from IPython.display import display, Image, HTML

## 2. Parameters

In [ ]:
PROJECT_ID = "ee-rnrimpact"

MAROANTSETRA_LON = 49.7333
MAROANTSETRA_LAT = -15.4333
AOI_BUFFER_KM = 220

# Raw Meteosat-9 IODC imagery is accessed through EUMETView, not as a standard
# public Earth Engine collection. Keep this metadata here for traceability.
METEOSAT9_IODC_PRODUCT = "MSG SEVIRI Indian Ocean Data Coverage"
METEOSAT9_OBSERVATION_MINUTES = 15
EUMETVIEW_URL = "https://user.eumetsat.int/data-access"

# Earth Engine scripted mode: Oya precipitation from geostationary VIS/IR sensors,
# including Meteosat-9/10. If you later ingest raw Meteosat into Earth Engine,
# set EARTH_ENGINE_METEOSAT_COLLECTION_ID to that ImageCollection id.
EARTH_ENGINE_METEOSAT_COLLECTION_ID = None
OYA_COLLECTION_ID = "projects/global-precipitation-nowcast/assets/global_estimation"

# Leave as None to auto-pick a likely cloud/IR band from the collection.
CLOUD_BAND = None
PREFERRED_CLOUD_BANDS = [
    "IR_108", "IR108", "IR_10_8", "IR_039", "IR_087", "IR_097", "IR_120", "IR_134",
    "WV_062", "WV_073", "VIS006", "VIS008", "HRV", "precipitation"
]

FLOOD_EVENTS = [
    {
        "name": "Cyclone Herold - Maroantsetra floods",
        "start": "2020-03-13T00:00:00",
        "end": "2020-03-18T23:59:59"
    },
    {
        "name": "Cyclone Gamane - north-east Madagascar floods",
        "start": "2024-03-26T00:00:00",
        "end": "2024-03-29T23:59:59"
    }
]

FRAME_INTERVAL_MINUTES = 15
FRAMES_PER_SECOND = 6
GIF_DIMENSIONS = 720
MAX_VIDEO_PIXELS = 26_214_400
SPLIT_EVENTS_BY_DAY = True
OUTPUT_DIR = "meteosat_cloud_animations"

# Set these to numbers if auto-stretch is not good for your selected band.
VIS_MIN = None
VIS_MAX = None

METEOSAT_PALETTE = ["111111", "333333", "666666", "999999", "cccccc", "ffffff"]
PRECIP_PALETTE = ["000096", "0064ff", "00b4ff", "33db80", "9beb4a", "ffeb00", "ffb300", "ff6400", "eb1e00", "af0000"]

## 3. Earth Engine setup and AOI

In [ ]:
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

center = ee.Geometry.Point([MAROANTSETRA_LON, MAROANTSETRA_LAT])
aoi = center.buffer(AOI_BUFFER_KM * 1000).bounds()

Map = geemap.Map()
Map.centerObject(center, 8)
Map.addLayer(aoi, {"color": "yellow"}, "Animation AOI")
Map.addLayer(center, {"color": "red"}, "Maroantsetra")
Map

## 4. Select Meteosat collection and band

In [ ]:
def try_collection(collection_id, start, end):
    collection = ee.ImageCollection(collection_id).filterDate(start, end).filterBounds(aoi)
    count = collection.size().getInfo()
    first = ee.Image(collection.first())
    bands = first.bandNames().getInfo() if count else []
    return collection, count, bands


def select_collection_for_events():
    first_event = FLOOD_EVENTS[0]
    if EARTH_ENGINE_METEOSAT_COLLECTION_ID:
        collection, count, bands = try_collection(
            EARTH_ENGINE_METEOSAT_COLLECTION_ID,
            first_event["start"],
            first_event["end"]
        )
        print("Using Earth Engine Meteosat collection:", EARTH_ENGINE_METEOSAT_COLLECTION_ID)
        print("Images in first event window:", count)
        print("Bands:", bands)
        return EARTH_ENGINE_METEOSAT_COLLECTION_ID, False, bands

    print("Raw Meteosat-9 IODC source:", METEOSAT9_IODC_PRODUCT)
    print("Raw Meteosat-9 access:", EUMETVIEW_URL)
    print("Earth Engine mode uses Oya geostationary precipitation.")
    print("Nominal Meteosat-9 cadence:", METEOSAT9_OBSERVATION_MINUTES, "minutes")

    try:
        collection, count, bands = try_collection(
            OYA_COLLECTION_ID,
            first_event["start"],
            first_event["end"]
        )
        print("Using Oya collection:", OYA_COLLECTION_ID)
        print("Images in first event window:", count)
        print("Bands:", bands)
        return OYA_COLLECTION_ID, True, bands
    except Exception as exc:
        raise RuntimeError("Oya collection is unavailable in this Earth Engine environment") from exc


ACTIVE_COLLECTION_ID, USING_PRECIP_FALLBACK, available_bands = select_collection_for_events()

if CLOUD_BAND is not None:
    selected_band = CLOUD_BAND
else:
    selected_band = next((b for b in PREFERRED_CLOUD_BANDS if b in available_bands), available_bands[0])

print("Selected band:", selected_band)

## 5. Animation helpers

In [ ]:
def event_collection(event):
    return (
        ee.ImageCollection(ACTIVE_COLLECTION_ID)
        .filterDate(event["start"], event["end"])
        .filterBounds(aoi)
    )


def auto_vis_range(collection, band):
    if VIS_MIN is not None and VIS_MAX is not None:
        return VIS_MIN, VIS_MAX

    first = ee.Image(collection.select(band).first())
    stats = first.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=aoi,
        scale=5000,
        bestEffort=True,
        maxPixels=1e7
    ).getInfo()

    lo = stats.get(f"{band}_p2")
    hi = stats.get(f"{band}_p98")

    if lo is None or hi is None or lo == hi:
        if USING_PRECIP_FALLBACK:
            return 0, 15
        return 180, 320

    return lo, hi


def split_event_by_day(event):
    if not SPLIT_EVENTS_BY_DAY:
        return [event]

    start = pd.Timestamp(event["start"])
    end = pd.Timestamp(event["end"])
    chunks = []

    for day_start in pd.date_range(start.normalize(), end.normalize(), freq="D"):
        chunk_start = max(start, day_start)
        chunk_end = min(end, day_start + pd.Timedelta(days=1) - pd.Timedelta(seconds=1))
        if chunk_start <= chunk_end:
            chunks.append({
                "name": f"{event['name']} - {chunk_start:%Y-%m-%d}",
                "start": chunk_start.strftime("%Y-%m-%dT%H:%M:%S"),
                "end": chunk_end.strftime("%Y-%m-%dT%H:%M:%S")
            })

    return chunks


def frame_count(event):
    start = pd.Timestamp(event["start"])
    end = pd.Timestamp(event["end"])
    minutes = max((end - start).total_seconds() / 60, FRAME_INTERVAL_MINUTES)
    return int(math.ceil(minutes / FRAME_INTERVAL_MINUTES))


def safe_dimensions(event):
    n_frames = frame_count(event)
    max_dim = int(math.floor(math.sqrt(MAX_VIDEO_PIXELS / max(n_frames, 1))))
    return max(64, min(GIF_DIMENSIONS, max_dim)), n_frames


def regular_frames(collection, event, band):
    start = ee.Date(event["start"])
    end = ee.Date(event["end"])
    n = end.difference(start, "minute").divide(FRAME_INTERVAL_MINUTES).ceil().int()
    empty = ee.Image.constant(0).rename(band).updateMask(ee.Image(0))

    def make_frame(i):
        i = ee.Number(i)
        d0 = start.advance(i.multiply(FRAME_INTERVAL_MINUTES), "minute")
        d1 = d0.advance(FRAME_INTERVAL_MINUTES, "minute")
        subset = collection.filterDate(d0, d1).select(band)
        img = ee.Image(ee.Algorithms.If(subset.size().gt(0), subset.mosaic(), empty))
        return img.set({
            "system:time_start": d0.millis(),
            "label": d0.format("YYYY-MM-dd HH:mm")
        })

    return ee.ImageCollection(ee.List.sequence(0, n.subtract(1)).map(make_frame))


def render_event_gif(event):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    collection = event_collection(event)
    count = collection.size().getInfo()
    if count == 0:
        print("No images for", event["name"])
        return None

    lo, hi = auto_vis_range(collection, selected_band)
    frames = regular_frames(collection, event, selected_band)
    dimensions, n_frames = safe_dimensions(event)

    palette = PRECIP_PALETTE if USING_PRECIP_FALLBACK else METEOSAT_PALETTE
    video_args = {
        "region": aoi,
        "dimensions": dimensions,
        "framesPerSecond": FRAMES_PER_SECOND,
        "min": lo,
        "max": hi,
        "palette": palette,
        "crs": "EPSG:3857"
    }

    safe_name = "".join(c if c.isalnum() else "_" for c in event["name"]).strip("_")
    out_gif = os.path.join(OUTPUT_DIR, f"{safe_name}.gif")
    url = frames.getVideoThumbURL(video_args)

    response = requests.get(url, timeout=120)
    response.raise_for_status()
    with open(out_gif, "wb") as f:
        f.write(response.content)

    print(event["name"])
    print("Raw images in event window:", count)
    print("Frames:", n_frames, "dimensions:", dimensions)
    print("Band:", selected_band, "min/max:", lo, hi)
    print("GIF:", out_gif)
    display(Image(filename=out_gif))
    return out_gif

## 6. Create animations

In [ ]:
gif_paths = []
for event in FLOOD_EVENTS:
    for event_chunk in split_event_by_day(event):
        path = render_event_gif(event_chunk)
        if path:
            gif_paths.append(path)

display(pd.DataFrame({"gif": gif_paths}))

## 7. Inspect one event on an interactive map

In [ ]:
EVENT_INDEX = 0

inspect_event = FLOOD_EVENTS[EVENT_INDEX]
inspect_collection = event_collection(inspect_event).select(selected_band)
lo, hi = auto_vis_range(inspect_collection, selected_band)

Map = geemap.Map()
Map.centerObject(center, 8)
Map.addLayer(aoi, {"color": "yellow"}, "Animation AOI")
Map.addLayer(
    inspect_collection.first(),
    {"min": lo, "max": hi, "palette": PRECIP_PALETTE if USING_PRECIP_FALLBACK else METEOSAT_PALETTE},
    inspect_event["name"]
)
Map.addLayer(center, {"color": "red"}, "Maroantsetra")
Map

## Notes

- Raw Meteosat-9 IODC imagery is the preferred source for true cloud motion around Madagascar. Use EUMETView for extraction: https://user.eumetsat.int/data-access
- For daytime cloud structure, use HRV when available. For day/night tracking, use the thermal infrared channel around 10.8 micrometers.
- In this Earth Engine notebook, Oya (`projects/global-precipitation-nowcast/assets/global_estimation`) is the automated alternative. It is precipitation, not raw cloud imagery, but it is derived from geostationary VIS/IR observations including Meteosat-9/10.
- MODIS (`MODIS/061/MOD02QKM`) and VIIRS can be used for high-resolution polar-orbit snapshots, but they do not provide continuous 15-minute motion loops.
- Earth Engine video thumbnails are limited by total pixels: `frames * width * height <= 26,214,400`. This notebook splits events by day and automatically lowers `dimensions` when needed.
- Adjust `FLOOD_EVENTS`, `AOI_BUFFER_KM`, `FRAME_INTERVAL_MINUTES`, `GIF_DIMENSIONS`, `SPLIT_EVENTS_BY_DAY`, and `CLOUD_BAND` as needed.